In [16]:
# Configuración
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum, desc

In [17]:
# Crear una sesión de Spark
# Es igual hacerlo todo en una línea, el "\" es solo para legibilidad, para avisar que se corta el código   
spark = SparkSession.builder \
    .appName("AnalisisVentas") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Leer CSV 
df = spark.read.csv("../data/online_retail.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5, truncate=False)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/10 8:26|2,55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/10 8:26|3,39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |12/1/10 8:26|2,7

## Correcciones y validaciones del dataframe

In [18]:
from pyspark.sql.functions import regexp_replace, col

# Tipo de columnas
df = df.withColumn("UnitPrice", regexp_replace(col("UnitPrice"), ",", ".")) # Reemplazar comas por puntos (separador decimal pyspark)
df = df.withColumn("UnitPrice", col("UnitPrice").cast("double"))

# Valores nulos
# df.na.drop()                        # Eliminar filas con nulos
df = df.na.fill({"Quantity": 0})            # Rellenar nulos
df.filter(df["UnitPrice"].isNull())     # Filas con nulos en una columna


DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: string, UnitPrice: double, CustomerID: int, Country: string]

In [19]:
# Número de filas
print(f"Total de registros: {df.count()}")

# Número de columnas
print(f"Total de columnas: {len(df.columns)}")

# Lista de columnas
print(df.columns)

# Tipo de columnas
print(df.dtypes)

Total de registros: 541909
Total de columnas: 8
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
[('InvoiceNo', 'string'), ('StockCode', 'string'), ('Description', 'string'), ('Quantity', 'int'), ('InvoiceDate', 'string'), ('UnitPrice', 'double'), ('CustomerID', 'int'), ('Country', 'string')]


In [20]:
df.describe().show() # estadísticas descriptivas básicas
# el .show() es para mostrar el resultado en consola de dataframe

+-------+------------------+------------------+--------------------+------------------+-------------+-----------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|          Quantity|  InvoiceDate|        UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+------------------+-------------+-----------------+------------------+-----------+
|  count|            541909|            541909|              540455|            541909|       541909|           541909|            406829|     541909|
|   mean|  559965.752026781|27623.240210938104|             20713.0|  9.55224954743324|         NULL| 4.61111362608925|15287.690570239585|       NULL|
| stddev|13428.417280796919| 16799.73762842769|                NULL|218.08115785023327|         NULL|96.75985306117953| 1713.600303321597|       NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|            -80995|1/10/11

In [21]:
# Limpiar DB dejando solo las ventas positivas
df = df.filter(df.Quantity > 0)

# Limpiar DB eliminando filas con nulos en InvoiceDate
df = df.filter(df.InvoiceDate.isNotNull())


In [22]:
from pyspark.sql.functions import col, count, sum, avg

num_paises = df.select("Country").distinct().count()
print(f"Número de países distintos: {num_paises}")


df.groupBy("Country").count().orderBy(col("count").desc()).show(5)

Número de países distintos: 38
+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|486286|
|       Germany|  9042|
|        France|  8408|
|          EIRE|  7894|
|         Spain|  2485|
+--------------+------+
only showing top 5 rows


In [23]:
df.groupBy("Country").count()\
    .orderBy(col("count")\
    .desc())\
    .show(10)

+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|486286|
|       Germany|  9042|
|        France|  8408|
|          EIRE|  7894|
|         Spain|  2485|
|   Netherlands|  2363|
|       Belgium|  2031|
|   Switzerland|  1967|
|      Portugal|  1501|
|     Australia|  1185|
+--------------+------+
only showing top 10 rows


## Selección y filtrado

In [24]:
df.select('Country','Quantity').show(3)       # Seleccionar columnas
df.filter(df.Quantity > 100).show(5)        # Filtrar filas
df.where(df.Country == "Spain").show(5) # Igual que filter()
df.distinct().count()                  # Contar valores únicos


+--------------+--------+
|       Country|Quantity|
+--------------+--------+
|United Kingdom|       6|
|United Kingdom|       6|
|United Kingdom|       8|
+--------------+--------+
only showing top 3 rows
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity| InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|   536378|    21212|PACK OF 72 RETROS...|     120|12/1/10 9:37|     0.42|     14688|United Kingdom|
|   536387|    79321|       CHILLI LIGHTS|     192|12/1/10 9:58|     3.82|     16029|United Kingdom|
|   536387|    22780|LIGHT GARLAND BUT...|     192|12/1/10 9:58|     3.37|     16029|United Kingdom|
|   536387|    22779|WOODEN OWLS LIGHT...|     192|12/1/10 9:58|     3.37|     16029|United Kingdom|
|   536387|    22466|FAIRY TALE COTTAG...|     432|12/1/10 9:58|     1.45|     16029|Un

526054

## Agrupaciones y resúmenes

In [25]:
df.groupBy("Country").count().show(5)                     # Conteo por grupo
df.groupBy("Country").agg(avg("Quantity"), sum("UnitPrice"))   # Agregaciones múltiples
df.orderBy(desc("Quantity")).show(5)                        # Ordenar descendente


+-------+-----+
|Country|count|
+-------+-----+
| Sweden|  451|
|Germany| 9042|
| France| 8408|
|Belgium| 2031|
|Finland|  685|
+-------+-----+
only showing top 5 rows
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   581483|    23843|PAPER CRAFT , LIT...|   80995|  12/9/11 9:15|     2.08|     16446|United Kingdom|
|   541431|    23166|MEDIUM CERAMIC TO...|   74215| 1/18/11 10:01|     1.04|     12346|United Kingdom|
|   578841|    84826|ASSTD DESIGN 3D P...|   12540|11/25/11 15:57|      0.0|     13256|United Kingdom|
|   542504|    37413|                NULL|    5568| 1/28/11 12:03|      0.0|      NULL|United Kingdom|
|   573008|    84077|WORLD WAR 2 GLIDE...|    4800|10/27/11 12:26|     0.21|     12901|United Kingdom|
+-------

In [26]:
# Primera y ultima fecha 
from pyspark.sql.functions import min, max
df.select(min("InvoiceDate"), max("InvoiceDate")).show()


+----------------+----------------+
|min(InvoiceDate)|max(InvoiceDate)|
+----------------+----------------+
|   1/10/11 10:32|     9/9/11 9:52|
+----------------+----------------+



## Operaciones con columnas

In [27]:
from pyspark.sql.functions import lit, when

#df = df.withColumn("IVA", df.UnitPrice * 0.21)               # Nueva columna
#df = df.withColumnRenamed("Sales", "Ventas")             # Renombrar
#df = df.drop("ColumnaInnecesaria")                       # Eliminar
df = df.withColumn("Etiqueta", when(df.Quantity > 100, "Alta").otherwise("Baja"))


* withColumn: Crear o modificar una columna existente.

In [30]:
# Earnings per country per year
from pyspark.sql.functions import year, month, col, sum as _sum # Extraer año, mes, referirse a columna, sumar (renombrar para evitar conflicto con función sum)
from pyspark.sql.functions import to_timestamp # Convertir string a timestamp
from pyspark.sql.functions import round, col

df = df.withColumn("InvoiceDate", to_timestamp("InvoiceDate", "M/d/yy H:mm")) # Convertir string a timestamp
df = df.withColumn("Revenue", col("Quantity") * col("UnitPrice")) # Crear columna de ganancias

df_result = df.groupBy(year("InvoiceDate").alias("Year"), "Country") \
              .agg(_sum("Revenue").alias("TotalRevenue"))

df_result = df_result.withColumn("TotalRevenue", round(col("TotalRevenue"), 2))
print(df_result.dtypes)
df_result.orderBy(desc("TotalRevenue")).show(10)

[('Year', 'int'), ('Country', 'string'), ('TotalRevenue', 'double')]
+----+--------------+------------+
|Year|       Country|TotalRevenue|
+----+--------------+------------+
|2011|United Kingdom|  8254828.98|
|2010|United Kingdom|   748268.98|
|2011|   Netherlands|   276661.86|
|2011|          EIRE|    273420.7|
|2011|       Germany|    213626.0|
|2011|        France|    200098.8|
|2011|     Australia|   137488.46|
|2011|         Spain|    59733.38|
|2011|   Switzerland|    55784.98|
|2011|       Belgium|    39386.43|
+----+--------------+------------+
only showing top 10 rows


In [31]:
# FINALIZAR SESION
spark.stop()